<a href="https://colab.research.google.com/github/ansari603089-del/codeorbit_intership/blob/main/Customer_Churn_Prediction_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import files

uploaded = files.upload()

df = pd.read_csv('MCA_Customer_Churn_Noisy_Dataset.csv')

Saving MCA_Customer_Churn_Noisy_Dataset.csv to MCA_Customer_Churn_Noisy_Dataset.csv


In [3]:
print("First 5 records:")
display(df.head())

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
df.info()

First 5 records:


,Customer_ID,Customer_Name,Age,City,Plan,Contract_Type,Tenure_Months,Monthly_Charge_INR,Total_Charges_INR,Internet_Service,Tech_Support,Payment_Method,Last_Login_Days_Ago,Churn
0,CUST0001,Priya Kumar,28.0,NaN,Basic,One year,51,2390.57,119782.94,DSL,Yes,Credit card,74,Yes
1,CUST0002,Priya Sharma,38.0,NaN,Premium,One year,23,1730.95,35348.85,Fiber optic,Yes,UPI,32,Yes
2,CUST0003,Rahul Patel,45.0,Pune,Standard,Monthly,57,INR 1453.46,77915.52,No,Yes,Cash,59,Yes
3,CUST0004,Arjun Khan,20.0,Pune,Standard,Monthly,14,1302.35,18049.20,Fiber optic,No,UPI,50,No
4,CUST0005,Sneha Khan,26.0,Vijayawada,Premium,Two year,6,NaN,3596.35,Fiber optic,Yes,Credit card,89,No


Dataset Shape: (510, 14)

Column Names:
Index(['Customer_ID', 'Customer_Name', 'Age', 'City', 'Plan', 'Contract_Type',
       'Tenure_Months', 'Monthly_Charge_INR', 'Total_Charges_INR',
       'Internet_Service', 'Tech_Support', 'Payment_Method',
       'Last_Login_Days_Ago', 'Churn'],
      dtype='object')

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer_ID          510 non-null    object 
 1   Customer_Name        510 non-null    object 
 2   Age                  497 non-null    object 
 3   City                 501 non-null    object 
 4   Plan                 502 non-null    object 
 5   Contract_Type        510 non-null    object 
 6   Tenure_Months        510 non-null    int64  
 7   Monthly_Charge_INR   502 non-null    object 
 8   Total_Charges_INR    510 non-null    float64
 9   Internet_

In [4]:
print("Missing Values:")
print(df.isnull().sum())

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

Missing Values:
Customer_ID             0
Customer_Name           0
Age                    13
City                    9
Plan                    8
Contract_Type           0
Tenure_Months           0
Monthly_Charge_INR      8
Total_Charges_INR       0
Internet_Service        0
Tech_Support            9
Payment_Method          6
Last_Login_Days_Ago     0
Churn                   0
dtype: int64

Total Missing Values:
53


In [5]:
print("Duplicate Records:", df.duplicated().sum())

Duplicate Records: 10


In [6]:
df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (500, 14)


In [7]:
# Remove leading and trailing spaces
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print("Extra spaces removed successfully")

Extra spaces removed successfully


In [8]:
# Correct Plan column
df['Plan'] = df['Plan'].replace({
    'basic': 'Basic',
    'STANDARD': 'Standard',
    'Premum': 'Premium'
})

# Correct Payment Method column
df['Payment_Method'] = df['Payment_Method'].replace({
    'upi': 'UPI',
    'Credit Card': 'Credit card',
    'debit card': 'Debit card',
    'Bank Transfer': 'Bank transfer'
})

print("Inconsistent values corrected")

Inconsistent values corrected


In [9]:
# Clean Age column
df['Age'] = (
    df['Age']
    .astype(str)
    .str.replace(' years', '', regex=False)
)

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# Clean Monthly Charges
df['Monthly_Charge_INR'] = (
    df['Monthly_Charge_INR']
    .astype(str)
    .str.replace('INR ', '', regex=False)
)

df['Monthly_Charge_INR'] = pd.to_numeric(
    df['Monthly_Charge_INR'], errors='coerce'
)

# Convert other numerical columns
numeric_cols = [
    'Tenure_Months',
    'Total_Charges_INR',
    'Last_Login_Days_Ago'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df.dtypes)

Customer_ID             object
Customer_Name           object
Age                    float64
City                    object
Plan                    object
Contract_Type           object
Tenure_Months            int64
Monthly_Charge_INR     float64
Total_Charges_INR      float64
Internet_Service        object
Tech_Support            object
Payment_Method          object
Last_Login_Days_Ago      int64
Churn                   object
dtype: object


In [10]:
# Invalid Age
df.loc[(df['Age'] < 18) | (df['Age'] > 100), 'Age'] = np.nan

# Invalid Tenure
df.loc[
    (df['Tenure_Months'] < 0) |
    (df['Tenure_Months'] > 72),
    'Tenure_Months'
] = np.nan

# Invalid Monthly Charges
df.loc[
    (df['Monthly_Charge_INR'] < 299) |
    (df['Monthly_Charge_INR'] > 2499),
    'Monthly_Charge_INR'
] = np.nan

# Invalid Total Charges
df.loc[df['Total_Charges_INR'] < 0, 'Total_Charges_INR'] = np.nan

# Invalid Last Login
df.loc[
    (df['Last_Login_Days_Ago'] < 0) |
    (df['Last_Login_Days_Ago'] > 90),
    'Last_Login_Days_Ago'
] = np.nan

print("Invalid values converted to missing values")

Invalid values converted to missing values


In [11]:
# Numerical columns
num_cols = df.select_dtypes(include=np.number).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values handled successfully")

print(df.isnull().sum())

Missing values handled successfully
Customer_ID            0
Customer_Name          0
Age                    0
City                   0
Plan                   0
Contract_Type          0
Tenure_Months          0
Monthly_Charge_INR     0
Total_Charges_INR      0
Internet_Service       0
Tech_Support           0
Payment_Method         0
Last_Login_Days_Ago    0
Churn                  0
dtype: int64


In [12]:
print("Plan Values:")
print(df['Plan'].value_counts())

print("\nPayment Methods:")
print(df['Payment_Method'].value_counts())

print("\nChurn Values:")
print(df['Churn'].value_counts())

Plan Values:
Plan
Standard    215
Basic       160
Premium     125
Name: count, dtype: int64

Payment Methods:
Payment_Method
Credit card      116
UPI              101
Cash              95
Bank transfer     95
Debit card        93
Name: count, dtype: int64

Churn Values:
Churn
No     337
Yes    163
Name: count, dtype: int64


In [13]:
print("Cleaned Dataset:")
display(df.head(10))

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())

print("\nDataset Information:")
df.info()

Cleaned Dataset:


,Customer_ID,Customer_Name,Age,City,Plan,Contract_Type,Tenure_Months,Monthly_Charge_INR,Total_Charges_INR,Internet_Service,Tech_Support,Payment_Method,Last_Login_Days_Ago,Churn
0,CUST0001,Priya Kumar,28.0,Bengaluru,Basic,One year,51.0,2390.570,119782.94,DSL,Yes,Credit card,74.0,Yes
1,CUST0002,Priya Sharma,38.0,Bengaluru,Premium,One year,23.0,1730.950,35348.85,Fiber optic,Yes,UPI,32.0,Yes
2,CUST0003,Rahul Patel,45.0,Pune,Standard,Monthly,57.0,1453.460,77915.52,No,Yes,Cash,59.0,Yes
3,CUST0004,Arjun Khan,20.0,Pune,Standard,Monthly,14.0,1302.350,18049.20,Fiber optic,No,UPI,50.0,No
4,CUST0005,Sneha Khan,26.0,Vijayawada,Premium,Two year,6.0,1444.825,3596.35,Fiber optic,Yes,Credit card,89.0,No
5,CUST0006,Rohan Sharma,52.0,Bengaluru,Standard,One year,52.0,806.970,38342.91,Fiber optic,Yes,Debit card,77.0,No
6,CUST0007,Pooja Singh,46.0,Hyderabad,Basic,Monthly,35.0,699.920,24148.13,No,No,Cash,62.0,No
7,CUST0008,Priya Khan,25.0,Bengaluru,Standard,Two year,17.0,2004.290,34465.79,No,No,Cash,77.0,No
8,CUST0009,Pooja Patel,62.0,Bengaluru,Premium,Monthly,61.0,1014.730,57838.90,DSL,Yes,Credit card,40.0,Yes
9,CUST0010,Rahul Patel,41.0,Bengaluru,Standard,Two year,28.0,2063.830,51455.90,Fiber optic,No,Bank transfer,62.0,No



Dataset Shape:
(500, 14)

Missing Values:
0

Duplicate Records:
0

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
Index: 500 entries, 0 to 499
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer_ID          500 non-null    object 
 1   Customer_Name        500 non-null    object 
 2   Age                  500 non-null    float64
 3   City                 500 non-null    object 
 4   Plan                 500 non-null    object 
 5   Contract_Type        500 non-null    object 
 6   Tenure_Months        500 non-null    float64
 7   Monthly_Charge_INR   500 non-null    float64
 8   Total_Charges_INR    500 non-null    float64
 9   Internet_Service     500 non-null    object 
 10  Tech_Support         500 non-null    object 
 11  Payment_Method       500 non-null    object 
 12  Last_Login_Days_Ago  500 non-null    float64
 13  Churn                500 non-null    object 
dtypes: flo

In [14]:
# Save cleaned dataset
df.to_csv('MCA_Customer_Churn_Cleaned.csv', index=False)

print("Cleaned dataset saved successfully!")

# Download cleaned dataset
files.download('MCA_Customer_Churn_Cleaned.csv')

Cleaned dataset saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
print("========== DATA CLEANING REPORT ==========")

print("Original Dataset Records: 510")
print("Cleaned Dataset Records:", len(df))

print("Missing Values After Cleaning:",
      df.isnull().sum().sum())

print("Duplicate Records After Cleaning:",
      df.duplicated().sum())

print("Total Columns:", len(df.columns))

print("==========================================")

========== DATA CLEANING REPORT ==========
Original Dataset Records: 510
Cleaned Dataset Records: 500
Missing Values After Cleaning: 0
Duplicate Records After Cleaning: 0
Total Columns: 14
